# M4S1 — HPC Applications in Science, Engineering and Society

## Five small workloads, one HPC way of thinking

**Learning loop:** **predict → run → observe → discuss**

This notebook uses small, self-contained teaching kernels to represent real HPC application patterns:

| Case | Real application examples | Teaching kernel |
|---|---|---|
| Climate & weather | WRF, CESM, MPAS | 2D grid update + resolution scaling |
| CFD & engineering | OpenFOAM, SU2 | iterative 2D Poisson/pressure kernel |
| Health & genomics | BWA, GATK | chunked sequence analysis |
| Molecular & materials | GROMACS, LAMMPS | pairwise Lennard-Jones forces |
| Finance | Monte Carlo risk | vectorized scenario simulation |

> **Important:** these are simplified teaching kernels, not replacements for production applications.  
> The objective is to recognize the **computational pattern, parallelism, bottleneck, and useful metric**.

## 0 — Check the environment

Before doing science, check **where** the notebook is running and which software environment it is using.

On an HPC system, the software stack matters: Python version, numerical libraries, compiler/runtime, scheduler, and available modules.

In [ ]:
import os, sys, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.grid"] = True

def shell(cmd):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return (result.stdout + result.stderr).strip()

print("Hostname:", shell("hostname"))
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("CPU cores visible:", os.cpu_count())
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Slurm sbatch available:", shutil.which("sbatch") is not None)

print("\nLoaded modules / module system (if available):")
print(shell("bash -lc 'if type module >/dev/null 2>&1; then module list 2>&1; else echo \"module command not visible in this shell\"; fi'"))

## The five-question case canvas

For every case, ask:

1. **DOMAIN / PROBLEM** — What are we trying to achieve?
2. **COMPUTATION & DATA** — What must be calculated or processed?
3. **PARALLEL PATTERN** — What can happen simultaneously?
4. **BOTTLENECK** — Compute? Memory? Communication? I/O?
5. **METRIC** — What does “fast enough” mean here?

We will use the same canvas five times.

# Case 1 — Climate & weather

### Predict before running
If we make a weather grid **twice as fine in each horizontal direction**, do you expect the computational work to:
- roughly stay the same,
- roughly double,
- or grow much faster?

A production weather model has many variables, vertical levels, physics packages, communication, and I/O.  
Here we use a simple 2D diffusion-style grid update to expose the basic **grid × timestep** pattern.

In [ ]:
def climate_initial(n=256, seed=0):
    y, x = np.mgrid[0:n, 0:n]
    rng = np.random.default_rng(seed)
    cx, cy = n * 0.45, n * 0.55
    sigma = n * 0.08
    field = np.exp(-((x-cx)**2 + (y-cy)**2) / (2*sigma**2))
    field += 0.05 * rng.random((n, n))
    return field

def climate_stepper(field, steps=120, alpha=0.18):
    u = field.copy()
    for _ in range(steps):
        old = u.copy()
        u[1:-1, 1:-1] = old[1:-1, 1:-1] + alpha * (
            old[2:, 1:-1] + old[:-2, 1:-1] +
            old[1:-1, 2:] + old[1:-1, :-2] -
            4 * old[1:-1, 1:-1]
        )
    return u

n = 320
initial = climate_initial(n)
t0 = time.perf_counter()
final = climate_stepper(initial, steps=160)
elapsed = time.perf_counter() - t0

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].imshow(initial, cmap="turbo")
ax[0].set_title("Initial synthetic weather field")
ax[0].axis("off")
ax[1].imshow(final, cmap="turbo")
ax[1].set_title(f"After 160 grid updates ({elapsed:.3f} s)")
ax[1].axis("off")
plt.tight_layout()
plt.show()

print(f"Grid: {n} × {n} = {n*n:,} cells")
print(f"Elapsed: {elapsed:.3f} s")

In [ ]:
sizes = [128, 256, 512, 768]
times = []

for n in sizes:
    u = climate_initial(n)
    t0 = time.perf_counter()
    _ = climate_stepper(u, steps=100)
    times.append(time.perf_counter() - t0)

plt.figure()
plt.plot(sizes, times, marker="o")
plt.xlabel("Grid width N  (N × N cells)")
plt.ylabel("Runtime (s)")
plt.title("Climate teaching kernel: cost grows with resolution")
plt.show()

for n, t in zip(sizes, times):
    print(f"{n:4d} × {n:<4d} -> {n*n:>8,} cells -> {t:.4f} s")

### Observe / discuss

**Case canvas**
- **Problem:** forecast weather / simulate climate at high resolution
- **Computation:** grid cells × variables × timesteps; often many ensemble members
- **Parallel pattern:** domain decomposition + ensemble parallelism
- **Likely bottlenecks:** communication, memory bandwidth, I/O
- **Useful metric:** **time-to-forecast / ensemble throughput**

**Discuss:** Why can a scientifically correct forecast still be operationally useless?

**Real-world mapping:** WRF, CESM and MPAS are vastly more sophisticated, but the same pressure remains: **more resolution → more computation and data → more need for parallel HPC**.

# Case 2 — CFD & engineering

### Predict before running
A CFD mesh is split across several MPI ranks.

**Question:** Which cells normally require communication with another rank?

The kernel below is a simplified iterative pressure/Poisson solve — a common numerical pattern inside CFD workflows.

In [ ]:
def cfd_poisson(n=220, iterations=300):
    p = np.zeros((n, n), dtype=float)
    source = np.zeros_like(p)
    source[n//4, n//4] = 100.0
    source[3*n//4, 3*n//4] = -100.0
    dx = 1.0 / (n - 1)

    for _ in range(iterations):
        old = p.copy()
        p[1:-1, 1:-1] = (
            old[1:-1, 2:] + old[1:-1, :-2] +
            old[2:, 1:-1] + old[:-2, 1:-1] -
            source[1:-1, 1:-1] * dx * dx
        ) / 4.0

        p[:, 0] = p[:, -1] = 0.0
        p[0, :] = p[-1, :] = 0.0

    return p

n = 240
t0 = time.perf_counter()
pressure = cfd_poisson(n, 350)
elapsed = time.perf_counter() - t0

plt.figure(figsize=(8, 5))
plt.imshow(pressure, origin="lower", cmap="coolwarm")
plt.colorbar(label="Synthetic pressure")
plt.title(f"Simplified CFD pressure kernel ({n}×{n}, {elapsed:.3f} s)")

plt.axvline(n/2, color="black", linewidth=2)
plt.axhline(n/2, color="black", linewidth=2)

plt.text(n*0.23, n*0.75, "Rank 0", ha="center", weight="bold")
plt.text(n*0.75, n*0.75, "Rank 1", ha="center", weight="bold")
plt.text(n*0.23, n*0.23, "Rank 2", ha="center", weight="bold")
plt.text(n*0.75, n*0.23, "Rank 3", ha="center", weight="bold")
plt.xlabel("Cells")
plt.ylabel("Cells")
plt.show()

In [ ]:
sizes = [100, 180, 260, 360]
times = []

for n in sizes:
    t0 = time.perf_counter()
    _ = cfd_poisson(n, 220)
    times.append(time.perf_counter() - t0)

plt.figure()
plt.plot(sizes, times, marker="o")
plt.xlabel("Mesh width N")
plt.ylabel("Runtime (s)")
plt.title("CFD teaching kernel: finer mesh costs more")
plt.show()

print("Think: with MPI decomposition, where would halo exchange occur?")

### Observe / discuss

**Case canvas**
- **Problem:** airflow around a car, wing, turbine, building, etc.
- **Computation:** mesh + repeated numerical updates
- **Parallel pattern:** domain decomposition; MPI between subdomains; OpenMP/threads inside a node
- **Likely bottlenecks:** halo communication, memory bandwidth, load imbalance
- **Useful metric:** **time-to-solution / design iterations per day**

**Discuss:** Why can adding MPI ranks eventually stop helping?

**Real-world mapping:** OpenFOAM, SU2 and commercial CFD solvers use much richer discretizations and physics, but domain decomposition and communication remain central HPC ideas.

# Case 3 — Health & genomics

### Predict before running
A sequencing study contains **many independent reads or samples**.

Which pattern is more natural first?
- spatial domain decomposition,
- or **data/task parallelism**?

We will generate synthetic reads, split them into chunks, and reduce partial results.

In [ ]:
def generate_reads(n_reads=25_000, read_len=120, seed=4):
    rng = np.random.default_rng(seed)
    bases = np.array(list("ACGT"))
    arr = rng.choice(bases, size=(n_reads, read_len), p=[0.30, 0.20, 0.20, 0.30])
    return ["".join(row) for row in arr]

def analyse_chunk(reads):
    base_counts = Counter()
    kmer_counts = Counter()
    for read in reads:
        base_counts.update(read)
        kmer_counts.update(read[i:i+3] for i in range(len(read)-2))
    return base_counts, kmer_counts

reads = generate_reads()
chunks = np.array_split(np.array(reads, dtype=object), 4)

t0 = time.perf_counter()
partials = [analyse_chunk(list(chunk)) for chunk in chunks]
elapsed = time.perf_counter() - t0

base_total = sum((p[0] for p in partials), Counter())
kmer_total = sum((p[1] for p in partials), Counter())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bases = ["A", "C", "G", "T"]
ax[0].bar(bases, [base_total[b] for b in bases])
ax[0].set_title("Base composition")
ax[0].set_ylabel("Count")

top = kmer_total.most_common(10)
ax[1].bar([x[0] for x in top], [x[1] for x in top])
ax[1].set_title("Top synthetic 3-mers")
ax[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Reads analysed: {len(reads):,}")
print(f"Four chunks -> map locally -> reduce partial counters")
print(f"Elapsed: {elapsed:.3f} s")

In [ ]:
single_bases, single_kmers = analyse_chunk(reads)

print("Base counts identical:", single_bases == base_total)
print("3-mer counts identical:", single_kmers == kmer_total)
print("\nParallel pattern:")
print("READ CHUNKS -> independent analysis -> REDUCE / MERGE")

### Observe / discuss

**Case canvas**
- **Problem:** identify biological information from sequencing data
- **Computation:** read processing → alignment/analysis → variant calling
- **Parallel pattern:** data parallelism + pipeline stages
- **Likely bottlenecks:** I/O, memory, pipeline imbalance
- **Useful metric:** **samples/day / turnaround time**

**Discuss:** Why does equal file size not always imply equal runtime?

**Real-world mapping:** tools such as BWA and GATK perform much more sophisticated algorithms, but large genomics workflows still expose **chunking, pipelines, data movement, and throughput** as HPC concerns.

# Case 4 — Molecular simulation & materials

### Predict before running
If every particle interacts with every other particle, what happens to the amount of work when the number of particles doubles?

We will compute simplified **Lennard-Jones pair forces**, one of the classic ideas used to teach molecular simulation.

In [ ]:
def make_particles(n=180, density=0.50, seed=2):
    box = math.sqrt(n / density)
    side = math.ceil(math.sqrt(n))
    spacing = box / side
    xs = (np.arange(side) + 0.5) * spacing
    points = np.array(np.meshgrid(xs, xs)).reshape(2, -1).T[:n]
    rng = np.random.default_rng(seed)
    points += rng.normal(scale=0.03 * spacing, size=points.shape)
    return points, box

def lennard_jones_forces(pos, box, cutoff=3.0):
    delta = pos[:, None, :] - pos[None, :, :]
    delta -= box * np.round(delta / box)

    r2 = np.sum(delta**2, axis=-1)
    np.fill_diagonal(r2, np.inf)

    mask = r2 < cutoff**2
    inv_r2 = np.zeros_like(r2)
    inv_r2[mask] = 1.0 / r2[mask]
    inv_r6 = inv_r2**3
    inv_r12 = inv_r6**2

    scalar = np.zeros_like(r2)
    scalar[mask] = 24.0 * (2.0*inv_r12[mask] - inv_r6[mask]) * inv_r2[mask]
    forces = np.sum(scalar[..., None] * delta, axis=1)
    return forces

pos, box = make_particles()
t0 = time.perf_counter()
forces = lennard_jones_forces(pos, box)
elapsed = time.perf_counter() - t0

mag = np.linalg.norm(forces, axis=1)
scale = np.percentile(mag, 85)
safe_forces = forces / (scale + 1e-12)

plt.figure(figsize=(7, 7))
plt.scatter(pos[:, 0], pos[:, 1], s=28)
plt.quiver(pos[:, 0], pos[:, 1], safe_forces[:, 0], safe_forces[:, 1],
           angles="xy", scale_units="xy", scale=1.5, width=0.003)
plt.xlim(0, box); plt.ylim(0, box)
plt.gca().set_aspect("equal")
plt.title(f"Simplified molecular force calculation ({len(pos)} particles)")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

print(f"Force calculation runtime: {elapsed:.4f} s")

In [ ]:
sizes = [100, 200, 400, 800]
times = []

for n in sizes:
    pos, box = make_particles(n)
    t0 = time.perf_counter()
    _ = lennard_jones_forces(pos, box)
    times.append(time.perf_counter() - t0)

plt.figure()
plt.plot(sizes, times, marker="o", label="Measured")
plt.xlabel("Particles")
plt.ylabel("Runtime (s)")
plt.title("Naive pair-force kernel: rapidly increasing work")
plt.legend()
plt.show()

for n, t in zip(sizes, times):
    print(f"{n:4d} particles -> {t:.5f} s")

### Observe / discuss

**Case canvas**
- **Problem:** predict how atoms/molecules behave
- **Computation:** force evaluation → update positions → repeat
- **Parallel pattern:** spatial decomposition + accelerators
- **Likely bottlenecks:** force computation, communication, memory
- **Useful metric:** **simulated time/day (e.g. ns/day) / time-to-solution**

**Discuss:** Why are GPUs attractive for force calculations?

**Real-world mapping:** GROMACS and LAMMPS add advanced neighbor lists, decomposition, vectorization, GPUs, communication and sophisticated physical models. The small kernel above is only the computational idea.

# Case 5 — Finance: Monte Carlo at scale

### Predict before running
Suppose one risk scenario takes 1 millisecond.

What changes when you need **10 million independent scenarios before a decision deadline**?

This is an example of **embarrassingly parallel work followed by a reduction/statistical summary**.

In [ ]:
def finance_monte_carlo(n=500_000, seed=7):
    rng = np.random.default_rng(seed)
    S0 = 100.0
    mu = 0.05
    sigma = 0.22
    T = 1.0

    z = rng.standard_normal(n)
    terminal = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z)
    pnl = terminal - S0
    var_95 = np.quantile(pnl, 0.05)
    return pnl, var_95

t0 = time.perf_counter()
pnl, var95 = finance_monte_carlo()
elapsed = time.perf_counter() - t0

plt.figure(figsize=(9, 4.8))
plt.hist(pnl, bins=90, alpha=0.85)
plt.axvline(var95, linestyle="--", linewidth=2,
            label=f"5th percentile = {var95:.2f}")
plt.xlabel("Profit / loss")
plt.ylabel("Scenarios")
plt.title("Monte Carlo risk distribution")
plt.legend()
plt.show()

print(f"Scenarios: {len(pnl):,}")
print(f"Runtime: {elapsed:.3f} s")
print(f"Scenario throughput: {len(pnl)/elapsed:,.0f} scenarios/s")

In [ ]:
N = 80_000
rng = np.random.default_rng(9)
z = rng.standard_normal(N)
S0, mu, sigma, T = 100.0, 0.05, 0.22, 1.0

t0 = time.perf_counter()
vec = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
loop = np.empty(N)
for i, zi in enumerate(z):
    loop[i] = S0 * math.exp((mu - 0.5*sigma**2)*T + sigma*math.sqrt(T)*zi)
t_loop = time.perf_counter() - t0

print(f"Vectorized NumPy: {t_vec:.4f} s")
print(f"Python loop:      {t_loop:.4f} s")
print(f"Observed ratio on this machine: {t_loop/max(t_vec,1e-12):.1f}×")
print("Results agree:", np.allclose(vec, loop))

### Observe / discuss

**Case canvas**
- **Problem:** estimate portfolio/risk outcomes before a decision deadline
- **Computation:** many independent scenarios
- **Parallel pattern:** embarrassingly parallel → aggregate/reduce
- **Likely bottlenecks:** throughput, data movement, aggregation
- **Useful metric:** **scenarios/second / time-to-decision**

**Discuss:** How is this fundamentally different from CFD?

- CFD subdomains often exchange boundary information repeatedly.
- Monte Carlo scenarios can often run independently until final aggregation.

# Compare the five cases

| Case | Main pattern | Likely bottleneck | Useful metric |
|---|---|---|---|
| Climate | domain + ensemble | communication / memory / I/O | time-to-forecast |
| CFD | domain decomposition | halos / memory / balance | time-to-solution |
| Genomics | data + pipeline | I/O / memory / imbalance | samples/day |
| Molecular | spatial + accelerator | compute / communication | simulated ns/day |
| Finance | independent tasks + reduction | throughput / aggregation | scenarios/s |

> **Peak FLOPS is a system metric. Success is an application metric.**

# Optional cluster extension — submit the five cases through Slurm

The interactive examples above are deliberately small.  
This section shows how the **same notebook can create a real batch job array**.

Each array task represents a different application case.

> Run this only if `sbatch` is available from the Jupyter environment.  
> If not, copy the generated files to your normal SSH/Slurm terminal.

In [ ]:
runner = '\nimport sys, time, math\nimport numpy as np\n\ncase = sys.argv[1]\nt0 = time.perf_counter()\n\nif case == "climate":\n    n = 768\n    u = np.random.default_rng(1).random((n,n))\n    alpha = 0.18\n    for _ in range(150):\n        old = u.copy()\n        u[1:-1,1:-1] = old[1:-1,1:-1] + alpha * (\n            old[2:,1:-1] + old[:-2,1:-1] +\n            old[1:-1,2:] + old[1:-1,:-2] -\n            4*old[1:-1,1:-1]\n        )\n    metric = f"cells={n*n:,}"\n\nelif case == "cfd":\n    n = 500\n    p = np.zeros((n,n))\n    b = np.zeros_like(p)\n    b[n//4,n//4] = 100.0\n    b[3*n//4,3*n//4] = -100.0\n    dx = 1/(n-1)\n    for _ in range(300):\n        old = p.copy()\n        p[1:-1,1:-1] = (\n            old[1:-1,2:] + old[1:-1,:-2] +\n            old[2:,1:-1] + old[:-2,1:-1] -\n            b[1:-1,1:-1]*dx*dx\n        ) / 4\n    metric = f"mesh={n}x{n}"\n\nelif case == "genomics":\n    rng = np.random.default_rng(3)\n    bases = np.array(list("ACGT"))\n    arr = rng.choice(bases, size=(120_000,120))\n    counts = {b:int(np.count_nonzero(arr == b)) for b in bases}\n    metric = f"reads={len(arr):,}; A={counts[\'A\']:,}"\n\nelif case == "molecular":\n    n = 1200\n    box = math.sqrt(n/0.5)\n    pos = np.random.default_rng(4).random((n,2))*box\n    delta = pos[:,None,:] - pos[None,:,:]\n    delta -= box*np.round(delta/box)\n    r2 = np.sum(delta**2,axis=-1)\n    np.fill_diagonal(r2,np.inf)\n    mask = r2 < 3.0**2\n    metric = f"particles={n:,}; neighbor_pairs={int(mask.sum()):,}"\n\nelif case == "finance":\n    rng = np.random.default_rng(5)\n    n = 4_000_000\n    z = rng.standard_normal(n)\n    terminal = 100*np.exp((0.05-0.5*0.22**2) + 0.22*z)\n    var5 = np.quantile(terminal-100,0.05)\n    metric = f"scenarios={n:,}; p5={var5:.3f}"\n\nelse:\n    raise SystemExit("Unknown case")\n\nelapsed = time.perf_counter()-t0\nprint(f"CASE={case} | {metric} | runtime={elapsed:.4f}s")\n'
Path("s16_case_runner.py").write_text(runner)
print("Wrote s16_case_runner.py")

In [ ]:
python_exe = sys.executable
workdir = str(Path.cwd())

sbatch_text = f'''#!/bin/bash -l
#SBATCH --job-name=s16_apps
#SBATCH --array=0-4
#SBATCH --cpus-per-task=1
#SBATCH --mem=1G
#SBATCH --time=00:05:00
#SBATCH --output=s16_case_%A_%a.out

CASES=(climate cfd genomics molecular finance)
CASE=${{CASES[$SLURM_ARRAY_TASK_ID]}}

echo "Host: $(hostname)"
echo "Case: $CASE"

if type module >/dev/null 2>&1; then
    module list 2>&1 || true
fi

cd "{workdir}"
"{python_exe}" s16_case_runner.py "$CASE"
'''

Path("s16_cases.sbatch").write_text(sbatch_text)
print(Path("s16_cases.sbatch").read_text())

In [ ]:
SUBMIT = False

if SUBMIT:
    if shutil.which("sbatch"):
        print(shell("sbatch s16_cases.sbatch"))
    else:
        print("sbatch is not available here. Submit s16_cases.sbatch from the normal cluster terminal.")
else:
    print("SUBMIT=False: no jobs submitted.")
    print("When ready, set SUBMIT=True and rerun this cell.")

In [ ]:
outputs = sorted(Path(".").glob("s16_case_*.out"))

if not outputs:
    print("No Slurm outputs found yet.")
else:
    for f in outputs:
        print(f"\n--- {f.name} ---")
        print(f.read_text().strip())

# Final discussion / exit question

Choose **one new application not used today**:

- energy-grid simulation,
- digital twin,
- earthquake modelling,
- astrophysics,
- AI training,
- another example of your choice.

Apply the five-question canvas:

**DOMAIN → COMPUTATION & DATA → PARALLEL PATTERN → BOTTLENECK → METRIC**

### Final takeaway

**HPC is not simply “a faster computer.”**  
It is a way to combine algorithms, parallelism, architecture, software and infrastructure so that a useful scientific or engineering result arrives at the required scale and time.